<a href="https://colab.research.google.com/github/jorgemx546/rutas-inundacion/blob/main/rutas_inundacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rutas de evacuación en zona inundada
**Actividad:** Cuando el camino más corto no es el mejor · Fundamentos de Inteligencia Artificial

**Equipo:** Jorge, Jarett, Danie y Erick

Un agente debe llegar desde una zona afectada hasta un refugio. El terreno tiene niveles de inundación con distinto riesgo, y algunas zonas están bloqueadas. Comparamos BFS, Dijkstra y A* para mostrar la diferencia entre encontrar *una* ruta y encontrar la ruta *conveniente*.

In [2]:
import heapq, time
from collections import deque
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# Niveles de terreno según el grado de inundación
SECO, BAJO, MEDIO, ALTO, BLOQUEADO = 0, 1, 2, 3, 4
# Riesgo de cada nivel (BLOQUEADO nunca se pisa, así que su riesgo no se usa)
RIESGO = np.array([0, 1, 3, 6, 0])
PESO_RIESGO = 3  # cuánto penaliza el riesgo en el costo

def crear_terreno():
    g = np.zeros((10, 10), dtype=int)
    g[4, :] = BLOQUEADO      # "río" que cruza el mapa
    g[4, 1] = ALTO           # cruce corto pero muy inundado
    g[4, 8] = SECO           # puente seguro pero lejano
    g[3, 0:3] = BAJO         # inundación leve antes del cruce
    g[5, 0:3] = MEDIO        # inundación media después del cruce
    return g

INICIO, REFUGIO = (0, 1), (9, 1)
grid = crear_terreno()

In [4]:
def vecinos(grid, pos):
    f, c = pos
    for df, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
        nf, nc = f + df, c + dc
        if 0 <= nf < grid.shape[0] and 0 <= nc < grid.shape[1] and grid[nf, nc] != BLOQUEADO:
            yield (nf, nc)

def costo_celda(grid, pos, peso):
    # Entrar a una celda cuesta 1 paso + penalización por riesgo
    return 1 + peso * RIESGO[grid[pos]]

def costo_ruta(grid, ruta, peso):
    # La celda de inicio no se cuenta: el agente ya está ahí
    return float(sum(costo_celda(grid, p, peso) for p in ruta[1:]))

def riesgo_ruta(grid, ruta):
    # Mismo criterio que el costo: se suman las celdas a las que se entra
    return int(sum(RIESGO[grid[p]] for p in ruta[1:]))

def reconstruir(padres, meta):
    ruta, actual = [], meta
    while actual is not None:
        ruta.append(actual)
        actual = padres[actual]
    return ruta[::-1]

def resultado(nombre, grid, padres, meta, peso, expandidos, t0, costo_interno=None):
    ms = (time.perf_counter() - t0) * 1000
    if meta not in padres:  # refugio inaccesible
        return dict(algoritmo=nombre, ruta=None, costo=float('inf'), pasos=None,
                    riesgo=None, expandidos=expandidos, tiempo_ms=ms)
    ruta = reconstruir(padres, meta)
    # Dijkstra y A* reportan el costo que acumularon durante la búsqueda.
    # El validador lo recalcula por separado: si no coinciden, hay un error.
    costo = costo_ruta(grid, ruta, peso) if costo_interno is None else float(costo_interno)
    return dict(algoritmo=nombre, ruta=ruta, costo=costo, pasos=len(ruta) - 1,
                riesgo=riesgo_ruta(grid, ruta), expandidos=expandidos, tiempo_ms=ms)

In [5]:
def bfs(grid, inicio, meta, peso=PESO_RIESGO):
    """Menos pasos. Ignora el riesgo al decidir; su costo se calcula después."""
    t0 = time.perf_counter()
    cola = deque([inicio]); padres = {inicio: None}; exp = 0
    while cola:
        actual = cola.popleft(); exp += 1
        if actual == meta: break
        for v in vecinos(grid, actual):
            if v not in padres:
                padres[v] = actual; cola.append(v)
    return resultado("BFS", grid, padres, meta, peso, exp, t0)

def _busqueda_costo(nombre, grid, inicio, meta, peso, usar_heuristica, costo_paso):
    t0 = time.perf_counter()
    # Heurística Manhattan: admisible porque cada paso cuesta al menos 1
    h = (lambda p: abs(p[0] - meta[0]) + abs(p[1] - meta[1])) if usar_heuristica else (lambda p: 0)
    g = {inicio: 0.0}; padres = {inicio: None}; cerrados = set(); exp = 0
    heap = [(h(inicio), 0, inicio)]; desempate = 1
    while heap:
        _, _, actual = heapq.heappop(heap)
        if actual in cerrados: continue
        cerrados.add(actual); exp += 1
        if actual == meta: break
        for v in vecinos(grid, actual):
            nuevo = g[actual] + costo_paso(grid, v, peso)
            if v not in g or nuevo < g[v]:
                g[v] = nuevo; padres[v] = actual
                heapq.heappush(heap, (nuevo + h(v), desempate, v)); desempate += 1
    return resultado(nombre, grid, padres, meta, peso, exp, t0, g.get(meta))

# costo_paso permite inyectar un error a propósito en la prueba adicional (celda 8)
def dijkstra(grid, inicio, meta, peso=PESO_RIESGO, costo_paso=costo_celda):
    return _busqueda_costo("Dijkstra", grid, inicio, meta, peso, False, costo_paso)

def a_estrella(grid, inicio, meta, peso=PESO_RIESGO, costo_paso=costo_celda):
    return _busqueda_costo("A*", grid, inicio, meta, peso, True, costo_paso)

ALGORITMOS = [bfs, dijkstra, a_estrella]

### Explicación: terreno y algoritmos (Jorge)

- **Terreno.** Cuadrícula de 10×10. Cada celda tiene un nivel: seco, inundación baja, media, alta o bloqueado. El riesgo de cada nivel es 0, 1, 3 y 6; las celdas bloqueadas no se pueden pisar.
- **Mapa de prueba.** Un río bloquea la fila 4. Hay dos formas de cruzarlo: un cruce corto con inundación alta en (4,1) y un puente seco pero lejano en (4,8).
- **Costo.** Entrar a una celda cuesta `1 + PESO_RIESGO × riesgo`, con `PESO_RIESGO = 3`.
- **BFS** busca la ruta con menos pasos y no mira el riesgo: encuentra *una* ruta.
- **Dijkstra** busca la ruta de menor costo total: encuentra la ruta *conveniente*.
- **A\*** hace lo mismo que Dijkstra, pero usa la distancia Manhattan para explorar menos. Esa heurística nunca sobreestima porque cada paso cuesta al menos 1, así que A* sigue encontrando el costo mínimo.
- **Resultado medido:** BFS cruza el agua en 9 pasos con costo 39; Dijkstra y A* van por el puente en 23 pasos con costo 23. A* expandió 75 nodos y Dijkstra 86.